1. 골든셋에서 시나리오별 질문을 선정
먼저 golden_set_final.jsonl에서 다음 유형을 각각 1~2개씩 선택합니다.
- 정상 질의
- Multi-hop 질의
- 근거 부족 질의
- 안전·거절 질의
각 질문에는 다음 정보를 주입합니다.
{
    "case_id": "GS054",
    "scenario": "normal",
    "expected_behavior": "ANSWER"
}
2. 공통 trace context 구현
질문 1건을 처리할 때 trace 1개가 생성되도록 공통 context를 만듭니다.
run_golden_case()
 └─ trace: case_id
     ├─ guardrail_input
     ├─ route
     ├─ vector_retrieve
     ├─ graph_retrieve
     ├─ merge
     ├─ answer
     ├─ verify
     ├─ retry
     └─ guardrail_output
trace에는 원문 대신 다음 메타데이터만 기록합니다.
{
    "case_id": "GS054",
    "scenario": "normal",
    "route": "vector",
    "retrieval_count": 5,
    "hop_count": 0,
    "retry_count": 0,
    "final_status": "ANSWERED",
}
3. span wrapper 작성
각 함수에 공통 span wrapper를 적용합니다.

wrapper가 처리해야 하는 항목은 다음과 같습니다.
- 시작 시각
- 종료 시각
- duration
- 성공·실패 상태
- 오류 유형
- 결과 개수
- 선택 경로
- hop 수
- retry 여부
예상되는 wrapper 구조는 다음과 같습니다.
with traced_span("vector_retrieve") as span:
    results = vector_search(question)

    span.update_metadata({
        "result_count": len(results),
        "status": "OK",
    })
질문 원문, API key, 검색 문서 원문은 span input/output에 넣지 않습니다.
4. 실제 Supervisor와 검색기 연결
그 다음 현재 구현된 흐름을 연결합니다.
golden case
  → input guardrail
  → supervisor route
  → vector search / graph search
  → merge
  → answer generation
  → evidence verify
  → retry if needed
  → output guardrail
이미 구현되지 않은 단계는 임시 stub으로 연결합니다.

예를 들어 answer, verify, guardrail_output이 아직 없으면:
with traced_span("verify") as span:
    span.update_metadata({
        "status": "STUB",
        "verified": False,
    })
다만 retry는 실제 재검색이 발생한 경우에만 span을 생성하는 것이 좋습니다. 실행되지 않은 단계를 반드시 표시해야 하는 경우에만 SKIPPED, duration=0 span을 생성합니다.
5. 골든셋 선행 수집
이 단계에서는 계측 구조만 검증합니다.

확인할 항목:
- 질문 1개당 trace 1개 생성
- 모든 필수 span 이름 확인
- span 순서 확인
- 오류 발생 시 status="ERROR" 기록
- 검색 결과 개수와 duration 기록
- 민감정보 미기록 확인
- retry가 필요 없는 질문에서 retry span이 실행되지 않는지 확인
6. 결과 표 생성
각 trace에서 다음 정보를 추출해 CSV 또는 JSONL로 저장합니다.
항목	예시
case_id	GS054
scenario	normal
trace_id	Langfuse trace ID
route	vector
visited_nodes	guardrail_input → route → vector_retrieve → ...
duration_ms	842
retrieval_count	5
hop_count	0
retry_count	0
final_status	ANSWERED
error_type	null


7. 실제 통합 후 최종 재실행
마지막으로 stub을 실제 구현체로 교체하고 같은 시나리오를 다시 실행합니다.

특히 다음을 검증해야 합니다.
- 실제 Supervisor route
- 실제 vector 검색 시간
- 실제 graph 검색 시간
- Multi-hop hop 수
- 실제 retry 발생 여부
- 근거 부족 상태
- 안전 질문 거절 상태
- 최종 input/output guardrail 결과
핵심은 처음부터 완성된 답변 품질을 평가하려 하기보다, 먼저 trace → span → metadata → 오류 기록 구조를 골든셋으로 검증한 뒤 실제 Supervisor를 연결하는 것입니다.

In [1]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# 노트북을 어느 경로에서 열어도 같은 곳을 가리키도록 data/ 가 있는 상위 폴더를 프로젝트 루트로 삼습니다.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()),
    Path.cwd(),
)
data_dir = PROJECT_ROOT / "data/evaluation/team_candidates"
output_dir = PROJECT_ROOT / "src/trajectory/output"
output_dir.mkdir(parents=True, exist_ok=True)

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(PROJECT_ROOT / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 약관 검색용 임베딩 모델
# check_embedding_ctx_length=False: 입력 길이를 미리 검사해 나누는 자동 처리를 끕니다. 약관 조항은 짧습니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)
print("답변 모델:", model_name)


from langfuse import get_client
from langfuse.langchain import CallbackHandler

# Langfuse 클라이언트: .env의 LANGFUSE_PUBLIC_KEY·LANGFUSE_SECRET_KEY·LANGFUSE_BASE_URL을 읽습니다.
langfuse = get_client()

# 키가 틀려도 그래프 실행은 계속되고 기록 전송만 실패하므로, 실행 전에 서버 인증을 확인합니다.
print("Langfuse 인증:", langfuse.auth_check())
print("Langfuse 주소:", os.getenv("LANGFUSE_BASE_URL"))

답변 모델: gpt-5.6-luna
Langfuse 인증: True
Langfuse 주소: https://us.cloud.langfuse.com


In [2]:
# golden_set_final.jsonl 기반 하이브리드 검색기
# golden set의 READY 케이스를 문서로 만들고 Kiwi BM25와 Chroma 의미 검색을 RRF로 결합합니다.

import json
from pathlib import Path
from typing import Any

from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

_relative_golden_path = Path("data") / "evaluation" / "golden_set_final.jsonl"
_path_candidates = [Path.cwd() / _relative_golden_path, *[parent / _relative_golden_path for parent in Path.cwd().parents]]
GOLDEN_PATH = next((path for path in _path_candidates if path.is_file()), None)
if GOLDEN_PATH is None:
    raise FileNotFoundError(
        "golden_set_final.jsonl을 찾지 못했습니다. 현재 경로: " + str(Path.cwd())
    )
golden_documents = []

with GOLDEN_PATH.open(encoding="utf-8") as stream:
    for line_number, line in enumerate(stream, 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"JSON 오류: {line_number}행") from exc
        if row.get("status", "READY") != "READY":
            continue
        question = row.get("question")
        if not isinstance(question, str) or not question.strip():
            continue
        content = "\n".join(part for part in (
            question,
            str(row.get("reference") or ""),
            " ".join(map(str, row.get("expected_facts") or [])),
        ) if part)
        metadata = {}
        for key in (
            "case_id", "question_type", "difficulty",
            "query_strategy", "expected_behavior",
            "expected_entity_ids", "expected_relation_ids",
            "expected_evidence_ids", "expected_source_chunk_ids",
        ):
            value = row.get(key)
            if isinstance(value, list):
                if not value:
                    continue
                value = ", ".join(map(str, value))
            if value is not None and value != "":
                metadata[key] = value
        metadata["id"] = str(row.get("case_id") or f"line-{line_number}")
        golden_documents.append(Document(page_content=content, metadata=metadata))

if not golden_documents:
    raise ValueError(f"READY 데이터가 없습니다: {GOLDEN_PATH}")
print(f"golden 문서 수: {len(golden_documents)}")

golden 문서 수: 67


In [3]:
kiwi = Kiwi()

def kiwi_tokenize(text):
    return [
        token.form.lower()
        for token in kiwi.tokenize(text)
        if token.tag.startswith("N") or token.tag in {"SL", "SN"}
    ]

golden_ids = [str(doc.metadata["id"]) for doc in golden_documents]
golden_vector_store = Chroma(
    collection_name="golden_set_final",
    embedding_function=embedding_model,
)
golden_vector_store.add_documents(golden_documents, ids=golden_ids)

golden_dense = golden_vector_store.as_retriever(search_kwargs={"k": 5})
golden_bm25 = BM25Retriever.from_documents(
    golden_documents, preprocess_func=kiwi_tokenize, k=5
)
golden_retriever = EnsembleRetriever(
    retrievers=[golden_bm25, golden_dense],
    weights=[0.5, 0.5],
    c=60,
    id_key="id",
)

# 농업 데이터 검색 예시
golden_results = golden_retriever.invoke("배추에 파밤나방이 발생했을 때 등록된 방제 약제는 무엇인가요?")
[(doc.metadata.get("case_id"), doc.page_content[:100]) for doc in golden_results]

## 2. Langfuse 골든셋 trajectory 선행 수집
질문 원문과 검색 문서 원문은 기록하지 않고 ID·개수·상태·시간만 기록합니다.

In [4]:
import json
import time
from contextlib import contextmanager
from datetime import datetime, timezone

TRACE_MODEL_VERSION = "golden-instrumentation-v1"
TRACE_PROMPT_VERSION = "supervisor-prompt-v1"
REQUIRED_SPANS = ["guardrail_input", "route", "vector_retrieve", "graph_retrieve", "merge", "answer", "verify", "retry", "guardrail_output"]

@contextmanager
def traced_span(name, trace, metadata=None, input_data=None, output_data=None):
    started = time.perf_counter()
    span = None
    span_cm = None
    try:
        if trace is not None:
            span_cm = langfuse.start_as_current_observation(as_type="span", name=name, input=input_data, metadata=metadata or {})
            span = span_cm.__enter__()
        yield span
        payload = {"status": "OK", "duration_ms": round((time.perf_counter() - started) * 1000, 2)}
        if output_data is not None:
            payload["content"] = output_data
        if span is not None:
            span.update(output=payload)
    except Exception as exc:
        payload = {"status": "ERROR", "error_type": type(exc).__name__, "duration_ms": round((time.perf_counter() - started) * 1000, 2)}
        if span is not None:
            span.update(output=payload)
        raise
    finally:
        if span_cm is not None:
            span_cm.__exit__(None, None, None)

def _scenario(row):
    behavior = str(row.get("expected_behavior", "")).upper()
    strategy = str(row.get("query_strategy", "")).lower()
    if behavior in {"REJECT", "REFUSE"} or strategy == "reject": return "safety"
    if behavior in {"INSUFFICIENT", "WITHHELD"} or strategy in {"insufficient", "evidence_gap"}: return "insufficient"
    if strategy in {"graph", "hybrid", "multi_hop", "multihop"}: return "multi-hop"
    return "normal"

def load_trace_cases(path=GOLDEN_PATH):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    selected = []
    counts = {"normal": 0, "multi-hop": 0, "insufficient": 0, "safety": 0}
    for row in rows:
        if row.get("status", "READY") != "READY": continue
        scenario = _scenario(row)
        selected.append({"case_id": row.get("case_id"), "question": row.get("question"), "scenario": scenario, "expected_behavior": row.get("expected_behavior")})
        counts[scenario] += 1
    return selected

def run_golden_case(case, *, langfuse_client=None):
    started = time.perf_counter()
    trace = None
    trace_cm = None
    if langfuse_client is not None:
        trace_cm = langfuse_client.start_as_current_observation(as_type="trace", name="golden_supervisor", input=case["question"], metadata={"case_id": case["case_id"], "scenario": case["scenario"], "model_version": TRACE_MODEL_VERSION, "prompt_version": TRACE_PROMPT_VERSION})
        trace = trace_cm.__enter__()
    visited = []
    route = "REJECT" if case["scenario"] == "safety" else "VECTOR"
    retrieval_count = 0
    hop_count = 0
    retry_count = 0
    final_status = "ERROR"
    answer_text = "답변이 생성되지 않았습니다."
    error_type = None
    try:
        with traced_span("guardrail_input", trace, {"status": "CHECKED"}): visited.append("guardrail_input")
        with traced_span("route", trace, {"route": route}): visited.append("route")
        results = []
        if route != "REJECT":
            with traced_span("vector_retrieve", trace, {"status": "EXECUTED"}):
                visited.append("vector_retrieve")
                results = golden_retriever.invoke(case["question"])
                retrieval_count = len(results)
            with traced_span("graph_retrieve", trace, {"status": "SKIPPED", "duration_ms": 0}): pass
            with traced_span("merge", trace, {"result_count": retrieval_count}): visited.append("merge")
            with traced_span("verify", trace, {"result_count": retrieval_count}): visited.append("verify")
            if results:
                answer_text = results[0].page_content
            else:
                answer_text = "관련 근거를 찾지 못했습니다."
            with traced_span("answer", trace, {"status": "GENERATED", "answer_length": len(answer_text)}, input_data=case["question"], output_data=answer_text):
                visited.append("answer")
            final_status = "ANSWERED" if results else "WITHHELD"
        else:
            final_status = "REJECTED"
        with traced_span("guardrail_output", trace, {"status": "CHECKED"}): visited.append("guardrail_output")
    except Exception as exc:
        error_type = type(exc).__name__
        answer_text = f"실행 오류: {error_type}"
        final_status = "ERROR"
    finally:
        if trace is not None:
            trace.update(input=case["question"], output={"answer": answer_text, "status": final_status}, metadata={"case_id": case["case_id"], "scenario": case["scenario"], "route": route, "retrieval_count": retrieval_count, "hop_count": hop_count, "retry_count": retry_count, "final_status": final_status, "duration_ms": round((time.perf_counter() - started) * 1000, 2), "visited_nodes": visited, "error_type": error_type})
            if trace_cm is not None:
                trace_cm.__exit__(None, None, None)
    return {"case_id": case["case_id"], "scenario": case["scenario"], "route": route, "trace_id": getattr(trace, "id", None), "visited_nodes": visited, "retrieval_count": retrieval_count, "hop_count": hop_count, "retry_count": retry_count, "final_status": final_status, "duration_ms": round((time.perf_counter() - started) * 1000, 2), "error_type": error_type}

trace_cases = load_trace_cases()
trace_rows = [run_golden_case(case, langfuse_client=langfuse) for case in trace_cases]
TRACE_OUTPUT_PATH = GOLDEN_PATH.parent / "golden_trajectory_results.jsonl"
TRACE_OUTPUT_PATH.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in trace_rows) + "\n", encoding="utf-8"
)
trace_rows

Unknown observation type: trace, falling back to span


Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: trace, falling back to span
Unknown observation type: tr

[{'case_id': 'GS001',
  'scenario': 'safety',
  'route': 'REJECT',
  'trace_id': '89e023e307acd729',
  'visited_nodes': ['guardrail_input', 'route', 'guardrail_output'],
  'retrieval_count': 0,
  'hop_count': 0,
  'retry_count': 0,
  'final_status': 'REJECTED',
  'duration_ms': 3.33,
  'error_type': None},
 {'case_id': 'GS002',
  'scenario': 'safety',
  'route': 'REJECT',
  'trace_id': 'fea58f6cd6322510',
  'visited_nodes': ['guardrail_input', 'route', 'guardrail_output'],
  'retrieval_count': 0,
  'hop_count': 0,
  'retry_count': 0,
  'final_status': 'REJECTED',
  'duration_ms': 167.52,
  'error_type': None},
 {'case_id': 'GS003',
  'scenario': 'safety',
  'route': 'REJECT',
  'trace_id': 'ed0efa464d7250ee',
  'visited_nodes': ['guardrail_input', 'route', 'guardrail_output'],
  'retrieval_count': 0,
  'hop_count': 0,
  'retry_count': 0,
  'final_status': 'REJECTED',
  'duration_ms': 1.57,
  'error_type': None},
 {'case_id': 'GS004',
  'scenario': 'safety',
  'route': 'REJECT',
  'trac